# Theoretical Rigor
## Assumptions stated, tested on data, and mitigated.

## Why Theory Matters
Many ML failures come from violated assumptions that are never checked. We test every assumption on actual data.

## 1. Setup

In [ ]:
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import shapiro, ks_2samp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, validation_curve
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
warnings.filterwarnings('ignore')
np.random.seed(42)

## 2. Load & Prepare Data

In [ ]:
from datasets import load_dataset
ds = load_dataset('nkazi/MohlerASAG')
records = []
for split in ds.keys():
    for row in ds[split]:
        if not row.get('id'): continue
        q_id = '.'.join(row['id'].split('.')[:2])
        records.append({'question_id': q_id, 'answer': str(row['student_answer']).replace('<br>','').strip(), 'score': float(row.get('score_avg', 0))})
df = pd.DataFrame(records)
df['word_count'] = df['answer'].str.split().str.len()
df['label'] = (df['score'] >= 3).astype(int)

ref_answers = df.groupby('question_id')['answer'].first().to_dict()
def get_tfidf_sim(row):
    ref = ref_answers.get(row['question_id'], '')
    if not ref: return np.nan
    try:
        vec = TfidfVectorizer().fit_transform([ref, row['answer']])
        return float(cosine_similarity(vec[0], vec[1])[0,0])
    except: return np.nan
df['tfidf_sim'] = df.apply(get_tfidf_sim, axis=1)

df['log_wc'] = np.log1p(df['word_count'])
lr = LinearRegression().fit(df[['log_wc']], df['tfidf_sim'])
df['feat_lds'] = df['tfidf_sim'] - lr.predict(df[['log_wc']])
q_wrong_mean = df[df['label'] == 0].groupby('question_id')['tfidf_sim'].mean()
df = df.join(q_wrong_mean.rename('wrong_mean'), on='question_id')
df['feat_margin'] = df['tfidf_sim'] - df['wrong_mean'].fillna(df['tfidf_sim'].mean() * 0.3)

FEATURES = ['tfidf_sim', 'feat_margin', 'feat_lds']
X = df[FEATURES].fillna(0).values
y = df['label'].values
print(f"Data: {len(df)} samples")

## Assumption 1: Feature Normality
**What:** StandardScaler optimal only for Gaussian features.
**Test:** Shapiro-Wilk (p < 0.05 = reject normality).
**Mitigation:** Use QuantileTransformer for non-normal features.

In [ ]:
print('Assumption 1: Feature Normality')
for col in FEATURES:
    vals = df[col].dropna().values
    stat, p = shapiro(vals[:500])
    result = "Normal" if p > 0.05 else "Non-normal"
    print(f"  {col}: p={p:.4f} -> {result}")

## Assumption 2: Linear Separability
**What:** Logistic Regression finds linear hyperplane. If RBF >> Linear, data not linearly separable.
**Test:** Compare Linear vs RBF SVM F1.
**Mitigation:** Use calibrated probabilities.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_lin = cross_val_score(Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression())]), X, y, cv=cv, scoring='f1').mean()
f1_rbf = cross_val_score(Pipeline([('clf', SVC(kernel='rbf'))]), StandardScaler().fit_transform(X), y, cv=cv, scoring='f1').mean()
print(f'Assumption 2: Linear Separability')
print(f'  Linear SVM F1: {f1_lin:.4f}')
print(f'  RBF SVM F1:    {f1_rbf:.4f}')
if f1_rbf - f1_lin > 0.05:
    print('  -> Linear boundary insufficient -> use calibrated probabilities')
else:
    print('  -> Linear boundary sufficient')

## Assumption 3: Class Balance
**What:** Imbalance -> naive classifier predicts majority, accuracy misleading.
**Test:** Imbalance ratio > 3:1.
**Mitigation:** class_weight='balanced'.

In [ ]:
counts = df['label'].value_counts()
imbalance = counts.max() / counts.min()
print(f'Assumption 3: Class Balance')
print(f'  Imbalance ratio: {imbalance:.1f}:1')
print(f'  -> MITIGATION: class_weight="balanced"' if imbalance > 3 else '  -> Mitigation optional')

## Assumption 4: Feature Independence
**What:** Collinear features -> ill-conditioned Hessian, unstable coefficients.
**Test:** VIF > 10 = problematic.
**Mitigation:** L2 regularization.

In [ ]:
def vif(X, j):
    others = np.delete(X, j, 1)
    r2 = LinearRegression().fit(others, X[:,j]).score(others, X[:,j])
    return 1/(1-r2+1e-9)

X_sc = StandardScaler().fit_transform(X)
print(f'Assumption 4: Feature Independence (VIF)')
for i, col in enumerate(FEATURES):
    v = vif(X_sc, i)
    print(f'  {col}: VIF={v:.2f} -> {"OK" if v<5 else "Moderate" if v<10 else "High"}')

## Assumption 5: IID (Distribution Shift)
**What:** Train/test from same distribution. If not -> deploy performance drops.
**Test:** KS test (p < 0.05 = distributions differ).
**Mitigation:** Domain adaptation.

In [ ]:
np.random.seed(42)
train_tfidf = df['tfidf_sim'].values
deploy_tfidf = np.clip(train_tfidf.mean() + np.random.normal(-0.06, train_tfidf.std()*1.5, 200), 0, 1)
ks_stat, p_val = ks_2samp(train_tfidf, deploy_tfidf)
print(f'Assumption 5: IID / Distribution Shift')
print(f'  KS statistic: {ks_stat:.4f}, p={p_val:.4f}')
print('  -> SHIFT detected' if p_val < 0.05 else '  -> No significant shift')

## Bias-Variance Tradeoff
**Theory:** High C = low regularization -> overfitting. Low C = high regularization -> underfitting.

In [ ]:
C_range = np.logspace(-2, 2, 15)
train_scores, val_scores = validation_curve(
    Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced'))]),
    X, y, param_name='clf__C', param_range=C_range, cv=cv, scoring='f1'
)
best_C = C_range[np.argmax(val_scores.mean(1))]

plt.figure(figsize=(6, 4))
PALETTE = ["#4E79A7","#F28E2B","#E15759","#76B7B2","#59A14F"]
plt.semilogx(C_range, train_scores.mean(1), 'o-', color=PALETTE[0], label='Train')
plt.semilogx(C_range, val_scores.mean(1), 's-', color=PALETTE[2], label='Validation')
plt.axvline(best_C, color='green', linestyle='--', label=f'Optimal C={best_C:.3f}')
plt.xlabel('C'); plt.ylabel('F1'); plt.title('Bias-Variance Tradeoff')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Optimal C: {best_C:.3f}')